# Task 3: Symbolic Recovery

Extract the learned dynamics from the trained Neural ODE and recover the SIR equations using:
- SINDy (sparse regression with physics-motivated library)
- PySR (genetic programming symbolic regression)
- Finite-difference baseline

In [ ]:
!git clone https://github.com/invi-bhagyesh/sira.git 2>/dev/null || echo "already cloned"
%cd sira
!pip install -q -r requirements.txt

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from src.models.neural_ode import NeuralODE
from src.symbolic.recovery import extract_derivatives, run_sindy, run_pysr
from src.simulation.dataset import load_dataset, train_val_test_split

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

In [ ]:
model = NeuralODE(hidden_dim=64, n_layers=3)
model.load_state_dict(torch.load('checkpoints/neural_ode.pt', map_location=device))
model.to(device)
model.eval()
print('Loaded trained Neural ODE')

## 1. Extract Derivatives from Neural ODE

Sample 10,000 state-parameter points uniformly and evaluate the learned f(s,i,r,beta,gamma) via autograd.

In [ ]:
states, params, derivs = extract_derivatives(model, n_points=10000, device=device)

print(f'Extracted derivatives at {len(states)} points')
print(f'States range: s=[{states[:,0].min():.2f}, {states[:,0].max():.2f}], '
      f'i=[{states[:,1].min():.2f}, {states[:,1].max():.2f}]')
print(f'Params range: beta=[{params[:,0].min():.2f}, {params[:,0].max():.2f}], '
      f'gamma=[{params[:,1].min():.2f}, {params[:,1].max():.2f}]')
print(f'\nDerivative statistics:')
for j, name in enumerate(['ds/dt', 'di/dt', 'dr/dt']):
    print(f'  {name}: mean={derivs[:,j].mean():.4f}, std={derivs[:,j].std():.4f}')

## 2. SINDy: Sparse Identification of Nonlinear Dynamics

Physics-motivated candidate library. Expect to recover:
- ds/dt = -beta * s * i
- di/dt = +beta * s * i - gamma * i
- dr/dt = +gamma * i

In [ ]:
sindy_results = run_sindy(states, params, derivs, threshold=0.05)

print('SINDy Recovered Equations:')
print('=' * 60)
for name, res in sindy_results.items():
    print(f'\n{name}:')
    print(f'  {res["equation"]}')
    print(f'  ({res["nonzero_terms"]} nonzero terms)')

print('\n' + '=' * 60)
print('\nGround truth:')
print('  ds/dt = -1.0 * beta*s*i')
print('  di/dt = +1.0 * beta*s*i - 1.0 * gamma*i')
print('  dr/dt = +1.0 * gamma*i')

In [ ]:
# coefficient accuracy
print('Coefficient comparison (nonzero terms only):')
print()

ground_truth = {
    'ds/dt': {'beta*s*i': -1.0},
    'di/dt': {'beta*s*i': 1.0, 'gamma*i': -1.0},
    'dr/dt': {'gamma*i': 1.0},
}

for name in ['ds/dt', 'di/dt', 'dr/dt']:
    print(f'{name}:')
    coeffs = sindy_results[name]['coefficients']
    gt = ground_truth[name]
    for term, true_val in gt.items():
        pred_val = coeffs.get(term, 0.0)
        err = abs(pred_val - true_val)
        print(f'  {term}: predicted={pred_val:+.4f}, true={true_val:+.1f}, error={err:.4f}')
    print()

## 3. PySR: Symbolic Regression

Genetic programming search over symbolic expressions. No pre-specified library needed.

In [ ]:
# PySR for ds/dt
print('Running PySR for ds/dt...')
pysr_ds = run_pysr(states, params, derivs, compartment='ds/dt', niterations=50)
print('\nBest equation for ds/dt:')
print(pysr_ds)

In [ ]:
# PySR for di/dt
print('Running PySR for di/dt...')
pysr_di = run_pysr(states, params, derivs, compartment='di/dt', niterations=50)
print('\nBest equation for di/dt:')
print(pysr_di)

In [ ]:
# PySR for dr/dt
print('Running PySR for dr/dt...')
pysr_dr = run_pysr(states, params, derivs, compartment='dr/dt', niterations=50)
print('\nBest equation for dr/dt:')
print(pysr_dr)

## 4. Finite-Difference Baseline

Apply SINDy directly to derivatives estimated from mean trajectories (no Neural ODE). This is noisier but serves as a comparison.

In [ ]:
data = load_dataset('data/sir_dataset.npz')
train, val, test = train_val_test_split(data)

t_grid = train['t_grid']
dt = t_grid[1] - t_grid[0]

# central differences on mean trajectories
ds_dt = np.gradient(train['mean_s'], dt, axis=1)
di_dt = np.gradient(train['mean_i'], dt, axis=1)
dr_dt = np.gradient(train['mean_r'], dt, axis=1)

# flatten: each (trajectory, time_point) is one data point
s_flat = train['mean_s'].flatten()
i_flat = train['mean_i'].flatten()
r_flat = (1.0 - train['mean_s'] - train['mean_i']).flatten()

n_traj, n_time = train['mean_s'].shape
beta_flat = np.repeat(train['params'][:, 0], n_time)
gamma_flat = np.repeat(train['params'][:, 1], n_time)

fd_states = np.column_stack([s_flat, i_flat, r_flat])
fd_params = np.column_stack([beta_flat, gamma_flat])
fd_derivs = np.column_stack([ds_dt.flatten(), di_dt.flatten(), dr_dt.flatten()])

# subsample to avoid huge arrays
rng = np.random.default_rng(42)
idx = rng.choice(len(fd_states), size=min(20000, len(fd_states)), replace=False)
fd_states, fd_params, fd_derivs = fd_states[idx], fd_params[idx], fd_derivs[idx]

fd_results = run_sindy(fd_states, fd_params, fd_derivs, threshold=0.05)

print('SINDy on Finite Differences (baseline):')
print('=' * 60)
for name, res in fd_results.items():
    print(f'\n{name}:')
    print(f'  {res["equation"]}')
    print(f'  ({res["nonzero_terms"]} nonzero terms)')

## 5. Comparison

Summary of all recovered equations across methods.

In [ ]:
print('Method Comparison')
print('=' * 70)
print(f'{"Method":<25} {"ds/dt":<20} {"di/dt":<25} {"dr/dt":<15}')
print('-' * 70)
print(f'{"Ground Truth":<25} {"-beta*s*i":<20} {"beta*s*i - gamma*i":<25} {"gamma*i":<15}')
print(f'{"SINDy (Neural ODE)":<25} {sindy_results["ds/dt"]["nonzero_terms"]:>3} terms{"":>12} {sindy_results["di/dt"]["nonzero_terms"]:>3} terms{"":>17} {sindy_results["dr/dt"]["nonzero_terms"]:>3} terms')
print(f'{"SINDy (Finite Diff)":<25} {fd_results["ds/dt"]["nonzero_terms"]:>3} terms{"":>12} {fd_results["di/dt"]["nonzero_terms"]:>3} terms{"":>17} {fd_results["dr/dt"]["nonzero_terms"]:>3} terms')
print('\nFull equations:')
for method, res in [('SINDy (Neural ODE)', sindy_results), ('SINDy (Finite Diff)', fd_results)]:
    print(f'\n  {method}:')
    for name in ['ds/dt', 'di/dt', 'dr/dt']:
        print(f'    {name} = {res[name]["equation"]}')